In [ ]:
#This is a test of the inversion process with MISMIP_PLUS
using WAVI 
using ImageFiltering
using SparseArrays
using IterativeSolvers, Plots, NCDatasets

In [ ]:
function mismip_plus_bed(x,y)
    xbar = 300000.0
    b0 = -150.0; b2 = -728.8; b4 = 343.91; b6 = -50.75;
    wc = 24000.0; fc = 4000.0; dc = 500.0;
    bx(x)=b0+b2*(x/xbar)^2+b4*(x/xbar)^4+b6*(x/xbar)^6;
    by(y)= dc*( (1+exp(-2(y-wc)/fc))^(-1) + (1+exp(2(y+wc)/fc))^(-1) );
    b = max(bx(x) + by(y), -720.0);
    return b;
end

In [ ]:
dx = 1.e3;
dy = 1.e3;
nx = round(Int, 640*1e3/dx);
ny = round(Int, 80*1e3/dx);
xx=[i*dx for i=1:nx, j=1:ny];
yy=[j*dy for i=1:nx, j=1:ny] .- 42000;
x = xx[:,1];
y = yy[1,:];

# Now we can plot
plt =  Plots.heatmap(x/1e3, y/1e3, mismip_plus_bed.(xx,yy)', 
                    xlabel = "x (km)", 
                    ylabel = "y (km)",
                    colorbar_title = "\n bed depth (m)",
                    right_margin = 4Plots.mm)
plot!(size = (600,400))

In [ ]:
dx = 8.e3;
dy = 8.e3;
nx = round(Int, 640*1e3/dx);
ny = round(Int, 80*1e3/dx); #fix the number of grid-cells in the x and y directions to match set extents

u_iszero = falses(nx+1,ny); #build x-direction velocity boundary condition matrix with no zero boundary conditions anywhere 
u_iszero[1,:].=true;        #set the x-direction velocity to zero at x = 0.
v_iszero=falses(nx,ny+1);   #build x-direction velocity boundary condition matrix with no zero boundary conditions anywhere 
v_iszero[:,1].=true;        #set the y-direction velocity to zero at y = 0 (free slip)
v_iszero[:,end].=true;       #set the y-direction velocity to zero at y = 84km (free slip)
v_iszero[1,:].=true;         #set the y-direction velocity to zero at x = 0km (no slip in combination with u_iszero)

grid = Grid(nx = nx, 
            ny = ny,   
            dx = dx, 
            dy = dy,
            u_iszero = u_iszero, 
            v_iszero = v_iszero)

In [ ]:
solver_params = SolverParams(maxiter_picard = 1)

initial_conditions = InitialConditions(initial_thickness = 100 .* ones(nx,ny));
params = Params( accumulation_rate = 0.3);

model = Model(grid = grid,
            bed_elevation = mismip_plus_bed, 
            initial_conditions = initial_conditions,
            params = params, 
            solver_params = solver_params)

In [ ]:
timestepping_params = TimesteppingParams(dt = 0.5, 
                                        end_time = 10000.,)

In [ ]:
folder = "MISMIP_PLUS_inversion_output/"
if isdir(folder)==false
    mkdir(folder)
end
output_params = OutputParams(outputs = (h = model.fields.gh.h, grfrac = model.fields.gh.grounded_fraction),
                            output_freq = 200.,
                            output_path = folder,
                            zip_format = "nc");
     

In [ ]:
simulation = Simulation(model = model, timestepping_params = timestepping_params, output_params = output_params)

In [ ]:
function MISMIP_PLUS_inversion(simulation)

    model = simulation.model

    # Read in data to be used for the inversion. This is taken from the MISMIP_PLUS output. 
    # The velocity data is calculated on the h-grid and returned on the u,v grid so doesn't need to be smoothed for use in the inversion.

    #pre-select points not near data gaps e.g. mask edge. Domain goes to ice shelf front so remove last row  
    #mask out first and last rows.
    speed_u_mask=deepcopy(simulation.model.fields.gu.mask)
    speed_u_mask[1:2, :] .= false
    speed_u_mask[end, :] .= false
    speed_v_mask=deepcopy(simulation.model.fields.gv.mask)
    speed_v_mask[:, 2] .= false
    speed_v_mask[:, end-1] .= false

    data = DataFields(grid = deepcopy(simulation.model.grid),
                              speed_u = deepcopy(simulation.model.fields.gu.us),
                              speed_u_mask =  speed_u_mask,
                              speed_v = deepcopy(simulation.model.fields.gv.vs),
                              speed_v_mask =  speed_v_mask,
                              dhdt = deepcopy(simulation.model.fields.gh.dhdt),
                              accumulation_rate = deepcopy(simulation.model.params.accumulation_rate),
                              dhdtacc_mask=deepcopy(simulation.model.fields.gh.mask))

    reltol=0.5
    abstol=0.1
    maxiter=500
    gmres_restart =50
    βgrounded_start=1.0e3
    βfloating_start=1.0e-10
    ηstart_guess = 1.0e6
    βpower = 0.1
    Bpower_shelf = 0.1
    Bpower_grounded = 0.0
    cg=false #?
    gmres=true #?

    inversion_params = InversionParams(reltol = reltol,
                                    maxiter = maxiter,
                                    abstol = abstol,
                                    gmres_restart = gmres_restart,
                                    βgrounded_start = βgrounded_start,
                                    βfloating_start = βfloating_start,
                                    ηstart_guess = ηstart_guess,
                                    βpower = βpower,
                                    Bpower_grounded = Bpower_grounded,
                                    Bpower_shelf = Bpower_shelf,
                                    cg = cg,
                                    gmres = gmres)
                     
    """
    JKV is the Kohn Vogelius cost function J
    """
    #JKVstepping parameters
    niter0 = 0
    #n_iter_out=500
    max_JKV_iterations = 5
    #n_iter_chkpt = 1000
    #n_iter_pchkpt= 100

    JKVstepping_params = JKVsteppingParams(niter0 = niter0, 
                                   #     n_iter_chkpt = n_iter_chkpt,
                                    #    n_iter_pchkpt = n_iter_pchkpt,
                                        n_iter_total = max_JKV_iterations)
                                      #  n_iter_out = n_iter_out)

    inversion_output = InversionOutput()
   
    dirichlet_model = deepcopy(simulation.model)

    inversion = Inversion(model = dirichlet_model,
                        data = data,
                        inversion_params=inversion_params,
                        inversion_output = inversion_output)
                            
    inversion_simulation = InversionSimulation(model = model, 
                                        inversion = inversion,
                                        JKVstepping_params = JKVstepping_params)
           
    run_inversion_simulation!(inversion_simulation)

    return inversion_simulation

end

In [ ]:
# inversion_results = MISMIP_PLUS_inversion(simulation)

In [ ]:
# copied from inside working function

model = simulation.model

# Read in data to be used for the inversion. This is taken from the MISMIP_PLUS output. 
# The velocity data is calculated on the h-grid and returned on the u,v grid so doesn't need to be smoothed for use in the inversion.

#pre-select points not near data gaps e.g. mask edge. Domain goes to ice shelf front so remove last row  
#mask out first and last rows.
speed_u_mask=deepcopy(simulation.model.fields.gu.mask)
speed_u_mask[1:2, :] .= false
speed_u_mask[end, :] .= false
speed_v_mask=deepcopy(simulation.model.fields.gv.mask)
speed_v_mask[:, 2] .= false
speed_v_mask[:, end-1] .= false

data = DataFields(grid = deepcopy(simulation.model.grid),
                        speed_u = deepcopy(simulation.model.fields.gu.us),
                        speed_u_mask =  speed_u_mask,
                        speed_v = deepcopy(simulation.model.fields.gv.vs),
                        speed_v_mask =  speed_v_mask,
                        dhdt = deepcopy(simulation.model.fields.gh.dhdt),
                        accumulation_rate = deepcopy(simulation.model.params.accumulation_rate),
                        dhdtacc_mask=deepcopy(simulation.model.fields.gh.mask))


reltol=0.5
abstol=0.1
maxiter=500
gmres_restart =50
βgrounded_start=1.0e3
βfloating_start=1.0e-10
ηstart_guess = 1.0e6
βpower = 0.1
Bpower_shelf = 0.1
Bpower_grounded = 0.0
cg=false #?
gmres=true #?

inversion_params = InversionParams(reltol = reltol,
                                maxiter = maxiter,
                                abstol = abstol,
                                gmres_restart = gmres_restart,
                                βgrounded_start = βgrounded_start,
                                βfloating_start = βfloating_start,
                                ηstart_guess = ηstart_guess,
                                βpower = βpower,
                                Bpower_grounded = Bpower_grounded,
                                Bpower_shelf = Bpower_shelf,
                                cg = cg,
                                gmres = gmres)
                    
"""
JKV is the Kohn Vogelius cost function J
"""
#JKVstepping parameters
niter0 = 0
#n_iter_out=500
max_JKV_iterations = 5
#n_iter_chkpt = 1000
#n_iter_pchkpt= 100

JKVstepping_params = JKVsteppingParams(niter0 = niter0, 
                                #     n_iter_chkpt = n_iter_chkpt,
                                #    n_iter_pchkpt = n_iter_pchkpt,
                                    n_iter_total = max_JKV_iterations)
                                    #  n_iter_out = n_iter_out)

inversion_output = InversionOutput()

dirichlet_model = deepcopy(simulation.model)

inversion = Inversion(model = dirichlet_model,
                    data = data,
                    inversion_params=inversion_params,
                    inversion_output = inversion_output)


In [ ]:
heatmap(inversion.model.fields.g3d.Φ[:,:,1]')